<a href="https://colab.research.google.com/github/czrclhynn/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/czrclhynn/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My lane as an ML task (type)

My chosen lane is **Refresh / Content Opportunity Scoring**.

I will frame this primarily as a **ranking/scoring task** because the practical question is not simply whether a page is declining, but **which pages should be reviewed first** when a content or SEO team has limited time.

The model would assign each page a priority score, then rank pages from highest to lowest priority. The output would support a review queue rather than automatically deciding that a page must be refreshed.

The decision this improves is: **Which pages should an editor or SEO specialist inspect first for possible content refresh?**

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Task type: Ranking / Scoring")
print("Lane: Refresh / Content Opportunity Scoring")

Task type: Ranking / Scoring
Lane: Refresh / Content Opportunity Scoring


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or proxy

For the starter dataset, I will use a **declining-performance indicator as a teaching proxy** for whether a page may deserve review.

The proxy is based on the observed `trend_direction` field. A page is treated as declining when `trend_direction == "down"`.

This is not the final production target. It is a convenient starter proxy that lets me test whether observable page-level signals can rank declining pages near the top of a review queue.

The target should come from an observed outcome rather than from a rule I invent myself. In a later version using the warehouse data, I would prefer a future performance outcome measured in a later time window.

Because `trend_direction` is derived from `trend_pct`, neither `trend_direction` nor `trend_pct` should be used as input features for a model predicting this proxy.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/czrclhynn/flyrank-ml-internship.git"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

print("Current folder:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded successfully.")
print("Rows:", len(df))

Current folder: /content/flyrank-ml-internship
Dataset loaded successfully.
Rows: 30000


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success metric

My primary success metric will be **Precision@50**.

Precision@50 answers a practical question: among the 50 pages ranked highest for review, what proportion are actually declining according to the starter proxy?

I chose this metric because the intended action is limited human review. A team may only have time to inspect a small number of pages, so performance at the top of the ranking matters more than overall classification accuracy.

A useful model should outperform a simple baseline rule at Precision@50. I would consider the approach promising if it consistently places a higher proportion of relevant pages in the top 50 than a fixed hand-written rule, especially on unseen data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    top_k_labels = np.asarray(labels)[order[:k]]
    return top_k_labels.mean()

# Simple hand-written baseline:
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["baseline_score"] = (
    stale *
    visible *
    df["impressions_90d"]
)

baseline_p50 = precision_at_k(
    df["baseline_score"],
    df["is_declining_proxy"],
    50
)

print(f"Baseline Precision@50: {baseline_p50:.3f}")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one content page**.

Each row in the starter dataset represents one pseudonymized content item. The columns contain page-level signals such as content age, update history, impressions, average search position, click-through rate, engagement, and word count.

For this lane, the ranking model would score each row independently and produce a page-level priority score.

The dataframe below shows the actual unit of analysis and a small set of candidate features together with the starter proxy target.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_columns = [
    "content_id",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "word_count",
    "is_declining_proxy"
]

lane_df = df[lane_columns].copy()

print("Unit of analysis: one row = one content page")
print("Shape:", lane_df.shape)

lane_df.head(10)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## 5. Why ML beats a fixed rule here

A fixed rule can capture simple intuition, such as prioritizing pages that are old and still receive many impressions. However, page performance depends on several signals at the same time, including content age, update recency, search visibility, average position, click-through rate, engagement, and page characteristics.

These signals may interact in ways that are difficult to express with a single if-statement. For example, an older page with many impressions may still be healthy, while a newer page with weaker engagement and declining visibility may deserve attention sooner.

Machine learning may earn its place if it can combine these signals into a more useful ranking than a simple fixed rule. The goal is not to replace human judgment, but to improve the order in which pages are reviewed.

If a fixed rule performs just as well or better on unseen data, then the simpler rule would be preferable. ML is only justified if it produces a measurable improvement in the decision.

In [ ]:
candidate_features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "word_count"
]

print("Candidate signals available for ranking:")
for feature in candidate_features:
    print("-", feature)

print("\nNumber of candidate signals:", len(candidate_features))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

- [x] I named the ML task type as ranking/scoring.
- [x] I defined a target/proxy and explained where it comes from.
- [x] I avoided using `trend_direction` and `trend_pct` as model features for the declining proxy.
- [x] I chose Precision@50 as a success metric and explained why it matches the action.
- [x] I showed the unit of analysis as a real dataframe.
- [x] One row represents one pseudonymized content page.
- [x] I connected the output to a real action: prioritizing pages for human review.
- [x] I explained why multiple interacting signals may justify ML over a fixed rule.
- [x] I acknowledged that a simple rule should be preferred if it performs just as well.
- [x] My claims are limited to observed, directional, and decision-support findings.
- [x] No private client names, URLs, or raw private queries are included.
- [x] The notebook is ready to run top to bottom and commit under `work/notebooks/`.